In [1]:
import numpy as np
import pandas as pd

cv = pd.concat([pd.read_csv("results/cv_logistic.csv"),
                pd.read_csv("results/cv_svm.csv"),
                pd.read_csv("results/cv_forest.csv")], ignore_index=True)

candidates = {
    "lasso":  ("logistic_L1", "C=0.1"),
    "svm":    ("svm_rbf", "C=1 gamma=0.001"),
    "forest": ("random_forest", "n_estimators=100 max_depth=10 max_features=sqrt"),
}

per_fold = pd.DataFrame({name: cv[(cv["model"] == m) & (cv["setting"] == s)].sort_values("fold")["macro_f1"].values
                         for name, (m, s) in candidates.items()})
print(per_fold.round(3))

summary = pd.DataFrame({"mean": per_fold.mean(), "se": per_fold.std() / np.sqrt(7)})
best = summary["mean"].idxmax()
threshold = summary.loc[best, "mean"] - summary.loc[best, "se"]
print(summary.round(4))
print("best:", best, " one-SE threshold:", round(threshold, 4))

for other in ["svm", "forest"]:
    diff = per_fold[other] - per_fold["lasso"]
    print(other, "minus lasso, per fold:", diff.round(3).values, " better on", (diff > 0).sum(), "of 7 folds")

   lasso    svm  forest
0  0.920  0.915   0.898
1  0.965  0.950   0.838
2  0.882  0.893   0.922
3  0.963  0.955   0.942
4  0.917  0.910   0.903
5  0.909  0.931   0.886
6  0.982  0.964   0.963
          mean      se
lasso   0.9341  0.0137
svm     0.9313  0.0101
forest  0.9075  0.0154
best: lasso  one-SE threshold: 0.9204
svm minus lasso, per fold: [-0.005 -0.015  0.011 -0.007 -0.008  0.022 -0.018]  better on 2 of 7 folds
forest minus lasso, per fold: [-0.021 -0.127  0.039 -0.02  -0.014 -0.023 -0.019]  better on 1 of 7 folds


## Decision (made before touching the test set)
- Candidates (each model's one-SE choice): lasso logistic regression C = 0.1 (0.934), RBF SVM C = 1, gamma = 0.001 (0.931), random forest 100 trees, depth 10, "sqrt" (0.907).
- One-SE rule across models: best = lasso, threshold 0.9204. Lasso and SVM qualify; the forest does not.
- Lasso vs SVM fold by fold: SVM better on 2 of 7 folds, differences between −0.018 and +0.022, so they are indistinguishable with 21 people.
- **Final model: lasso logistic regression, C = 0.1**, the simpler of the two (linear, ~180 features, predictions decomposable into feature contributions).
- Leave-one-person-out is not run: the 7-fold out-of-fold predictions already give a score for every training person (from models trained on 18 people instead of 20), and 21 extra saga fits (~30 min) would add little.
- The final model is trained on all 21 training people and evaluated **once** on the 9 test people.

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from config import SEED, DATA

X_train = pd.read_csv(DATA + "/train/X_train.txt", sep=r"\s+", header=None).values
y_train = np.loadtxt(DATA + "/train/y_train.txt", dtype=int)
X_test = pd.read_csv(DATA + "/test/X_test.txt", sep=r"\s+", header=None).values
y_test = np.loadtxt(DATA + "/test/y_test.txt", dtype=int)
person_test = np.loadtxt(DATA + "/test/subject_test.txt", dtype=int)

final_model = make_pipeline(StandardScaler(),
                            LogisticRegression(l1_ratio=1, solver="saga", C=0.1, max_iter=5000))
final_model.fit(X_train, y_train)

P_test = final_model.predict_proba(X_test)
pred_test = final_model.classes_[P_test.argmax(axis=1)]

print("test people:", sorted(set(person_test)), " windows:", len(y_test))
print("TEST macro-F1:", round(f1_score(y_test, pred_test, average="macro"), 4))
print("TEST accuracy:", round(accuracy_score(y_test, pred_test), 4))

test_out = pd.DataFrame(P_test, columns=["p_walk", "p_up", "p_down", "p_sit", "p_stand", "p_lie"])
test_out["true"] = y_test
test_out["pred"] = pred_test
test_out["person"] = person_test
test_out.to_csv("results/test_predictions_lasso.csv", index=False)

test people: [np.int64(2), np.int64(4), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(18), np.int64(20), np.int64(24)]  windows: 2947
TEST macro-F1: 0.9447
TEST accuracy: 0.9457


In [3]:
rng = np.random.default_rng(SEED)
people = np.unique(person_test)
rows_of = {p: np.where(person_test == p)[0] for p in people}

per_person = pd.Series({int(p): f1_score(y_test[r], pred_test[r], average="macro") for p, r in rows_of.items()})
print(per_person.round(3))

boot = []
for _ in range(2000):
    sample = rng.choice(people, size=len(people), replace=True)
    rows = np.concatenate([rows_of[p] for p in sample])
    boot.append(f1_score(y_test[rows], pred_test[rows], average="macro"))

low, high = np.percentile(boot, [2.5, 97.5])
print("95% bootstrap interval over people:", round(low, 3), "to", round(high, 3))
per_person.to_csv("results/test_per_person_lasso.csv", header=["macro_f1"])

2     0.983
4     0.954
9     0.881
10    0.826
12    0.969
13    0.990
18    0.959
20    0.895
24    1.000
dtype: float64
95% bootstrap interval over people: 0.906 to 0.975


In [4]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

others = {
    "svm": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, gamma=0.001)),
    "forest": RandomForestClassifier(n_estimators=100, max_depth=10, max_features="sqrt",
                                     random_state=SEED, n_jobs=-1),
}
for name, m in others.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    print(name, " test macro-F1:", round(f1_score(y_test, p, average="macro"), 4))

svm  test macro-F1: 0.947
forest  test macro-F1: 0.9194


# Part 7 summary: model selection and the final test

## Choosing the model (21 training people, the same 7 person-wise folds for every model)
| Fold (people) | Lasso C = 0.1 | SVM C = 1, γ = 0.001 | Forest 100 trees, depth 10 |
|---|---|---|---|
| 0 (15, 22, 25) | 0.920 | 0.915 | 0.898 |
| 1 (3, 8, 21) | 0.965 | 0.950 | **0.838** |
| 2 (14, 26, 29) | **0.882** | 0.893 | 0.922 |
| 3 (1, 6, 30) | 0.963 | 0.955 | 0.942 |
| 4 (7, 19, 28) | 0.917 | 0.910 | 0.903 |
| 5 (5, 16, 27) | 0.909 | 0.931 | 0.886 |
| 6 (11, 17, 23) | 0.982 | 0.964 | 0.963 |
| **Mean (SE)** | **0.934 (0.014)** | 0.931 (0.010) | 0.907 (0.015) |

- One-SE rule across models: threshold 0.934 − 0.014 = 0.920. **Lasso and SVM qualify; the forest does not.**
- Lasso vs SVM fold by fold: SVM better on 2 of 7 folds, differences −0.018 to +0.022, so they are indistinguishable.
- **Final model: lasso logistic regression, C = 0.1**, the simpler one (linear, ~180 features). The decision was written down before the test set was opened.
- "Hard people" depend partly on the model: fold 2 is the linear models' worst fold but the forest's best; fold 1 is the forest's worst.
- Leave-one-person-out was not run: the 7-fold out-of-fold predictions already score every training person, and 21 more slow saga fits would add little.

## The final test (9 test people, 2,947 windows, used once)
- Trained on all 21 training people (7,352 windows); scaling learned from training only.
- **Test macro-F1 = 0.945, accuracy = 0.946** (cross-validation estimate: 0.934; majority baseline: 0.05).
- **95% bootstrap interval over people: 0.906 to 0.975** (2,000 resamples of the 9 *people*, not windows, because a person's windows are not independent).
- Per person: 24: 1.000 · 13: 0.990 · 2: 0.983 · 12: 0.969 · 18: 0.959 · 4: 0.954 · 20: 0.895 · 9: 0.881 · **10: 0.826**. Most people are near-perfect; a few are clearly harder.

## Context only (run after the decision, not used for choosing)
- SVM test macro-F1 = 0.947; forest = 0.919. SVM vs lasso differ by 0.002, far inside the ±0.035 uncertainty. The forest trails, as in cross-validation.

## Takeaways
1. The cross-validation estimate (0.934) was reliable: the test result (0.945) is close and inside the expected range.
2. With 9 test people the honest claim is a range, roughly 0.91–0.97, not a precise number.
3. Lasso and SVM cannot be separated with this many people; the choice followed a rule fixed in advance.
4. The average hides a real spread between people (0.83–1.00), which Part 8 investigates.
